<a href="https://colab.research.google.com/github/minjaaaa/Schneider-Electric-Internship/blob/main/05_prompt_versions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
%cd /content
!rm -rf repo
!git clone https://github.com/minjaaaa/Schneider-Electric-Internship.git repo
%cd repo
!git log -1 --format="%h %ci %s"
!pip install -q python-dotenv bitsandbytes accelerate

/content
Cloning into 'repo'...
remote: Enumerating objects: 194, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 194 (delta 96), reused 134 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (194/194), 1.31 MiB | 23.05 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/repo
b9cb477 2026-10-17 19:54:48 +0200 Added four generative prompts version


In [17]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [18]:
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
VERSIONS = ["v1", "v2a", "v2b", "v2c"]          # v1 = kontrola: mora da ponovi bazno pokretanje
BASE_RUN = "2026-10-14_1022_gen_hybrid_qwen7b"  # odavde se uzima kontekst (context_ids) za svako pitanje
MAX_NEW_TOKENS = 700                            # v2c piše i dokaze, pa treba više mesta
MODEL_TAG = "qwen7b"

Zamrznut kontekst: isti chunk-ovi kao u baznom pokretanju

In [19]:
import sys, json
sys.path.insert(0, "src")
from chunking import load_jsonl
from config import DATA_PROCESSED, EVAL_QUESTIONS, RESULTS_DIR
from figures import figure_chunks

questions = json.loads(EVAL_QUESTIONS.read_text(encoding="utf-8"))
by_id = {c["chunk_id"]: c for c in load_jsonl(DATA_PROCESSED / "chunks_section.jsonl") + figure_chunks()}
base = {r["id"]: r for r in map(json.loads, open(RESULTS_DIR / BASE_RUN / "answers.jsonl", encoding="utf-8"))}
retrieved = {qid: [by_id[cid] for cid in r["context_ids"]] for qid, r in base.items()}
print(len(questions), "pitanja;", sum(len(v) for v in retrieved.values()), "chunk-ova u kontekstu")

49 pitanja; 245 chunk-ova u kontekstu


In [22]:
from src.generate import HFChatModel, answer_question, prompt_fingerprint
llm = HFChatModel(GEN_MODEL, max_new_tokens=MAX_NEW_TOKENS)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Drive (rezultati se kopiraju posle SVAKE verzije)

In [23]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/northstar_results"
os.makedirs(DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


 Pokretanje svih verzija

In [24]:
import shutil
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm

outs = {}
for version in VERSIONS:
    rows = []
    for q in tqdm(questions, desc=version):
        r = answer_question(q["question"], retrieved[q["id"]], llm, prompt_version=version)
        rows.append({"id": q["id"], "category": q["category"], "question": q["question"],
                     "expected_answer": q["expected_answer"], "sources": q["sources"], **r})
    out = Path("results") / f"{datetime.now():%Y-%m-%d_%H%M}_gen_prompt-{version}_{MODEL_TAG}"
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "answers.jsonl", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    json.dump({"generator": GEN_MODEL, "top_k": 5,
               "retriever": "8_hybrid (exclude_superseded, with_path, figures)",
               "context_from": BASE_RUN, "prompt_version": version,
               "prompt_hash": prompt_fingerprint(version), "max_new_tokens": MAX_NEW_TOKENS},
              open(out / "run_config.json", "w"), indent=2)
    shutil.copytree(out, f"{DRIVE_DIR}/{out.name}", dirs_exist_ok=True)
    outs[version] = out
    print(version, "->", out.name, "| parse greške:", sum(r["parse_error"] for r in rows),
          "| popravljeno:", sum(r.get("repaired", False) for r in rows))

v1:   0%|          | 0/49 [00:00<?, ?it/s]

v1 -> 2026-10-17_1829_gen_prompt-v1_qwen7b | parse greške: 0 | popravljeno: 0


v2a:   0%|          | 0/49 [00:00<?, ?it/s]

v2a -> 2026-10-17_1833_gen_prompt-v2a_qwen7b | parse greške: 0 | popravljeno: 0


v2b:   0%|          | 0/49 [00:00<?, ?it/s]

v2b -> 2026-10-17_1837_gen_prompt-v2b_qwen7b | parse greške: 0 | popravljeno: 0


v2c:   0%|          | 0/49 [00:00<?, ?it/s]

v2c -> 2026-10-17_1845_gen_prompt-v2c_qwen7b | parse greške: 0 | popravljeno: 1


Kontrola: da li v1 ponavlja bazno pokretanje

In [25]:
v1 = {r["id"]: r for r in map(json.loads, open(outs["v1"] / "answers.jsonl", encoding="utf-8"))}
diff = [qid for qid in v1 if v1[qid]["raw"].strip() != base[qid]["raw"].strip()]
print(f"v1 = bazno: {len(v1) - len(diff)}/{len(v1)} identičnih; razlikuju se: {diff}")

v1 = bazno: 49/49 identičnih; razlikuju se: []


Preuzimanje

In [26]:
!zip -qr results_prompts.zip {" ".join(str(o) for o in outs.values())}
from google.colab import files
files.download("results_prompts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>